In [1]:
from collections import defaultdict
from math import cos, hypot, radians, sin, sqrt, tan
from pathlib import Path
from xml.etree import ElementTree as ET
from zipfile import ZipFile

LANES_FILE = Path("movcud-zolaqlar-velo.kmz")
PARKING_FILE = Path("mpv-v-mps.-son.kmz")
missing = [p for p in (LANES_FILE, PARKING_FILE) if not p.is_file()]
if missing:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("KMZ fayllarını notebook ilə eyni qovluğa qoyun.") from exc
    files.upload()
    if any(not p.is_file() for p in missing):
        raise FileNotFoundError("Orijinal adlı KMZ faylları yüklənməyib.")

NS = {"k": "http://www.opengis.net/kml/2.2"}

def placemarks(path, layer):
    with ZipFile(path) as archive:
        kml = [name for name in archive.namelist() if name.lower().endswith(".kml")]
        if len(kml) != 1:
            raise ValueError(f"{path.name}: bir KML faylı gözlənilir.")
        root = ET.fromstring(archive.read(kml[0]))
    containers = [node for node in root.findall(".//k:Folder", NS) + root.findall(".//k:Document", NS)
                  if (node.findtext("k:name", default="", namespaces=NS) or "").strip() == layer]
    if len(containers) != 1:
        raise ValueError(f"{layer}: bir uyğun qat gözlənilir.")
    return containers[0].findall("k:Placemark", NS)

def coordinates(node):
    return [tuple(map(float, item.split(",")[:2])) for item in (node.text or "").split()]

# WGS84 koordinatları EPSG:32639 (UTM 39N) sisteminə çevrilir; nəticə metrlədir.
def utm39(lon, lat):
    a, f, k = 6378137.0, 1 / 298.257223563, 0.9996
    e2 = f * (2 - f)
    ep2 = e2 / (1 - e2)
    phi, delta = radians(lat), radians(lon - 51)
    s, c, t = sin(phi), cos(phi), tan(phi)
    n = a / sqrt(1 - e2 * s * s)
    T, C, A = t * t, ep2 * c * c, delta * c
    m = a * ((1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256) * phi
             - (3 * e2 / 8 + 3 * e2**2 / 32 + 45 * e2**3 / 1024) * sin(2 * phi)
             + (15 * e2**2 / 256 + 45 * e2**3 / 1024) * sin(4 * phi)
             - 35 * e2**3 / 3072 * sin(6 * phi))
    x = 500000 + k * n * (A + (1 - T + C) * A**3 / 6
         + (5 - 18 * T + T * T + 72 * C - 58 * ep2) * A**5 / 120)
    y = k * (m + n * t * (A * A / 2 + (5 - T + 9 * C + 4 * C * C) * A**4 / 24
         + (61 - 58 * T + T * T + 600 * C - 330 * ep2) * A**6 / 720))
    return x, y

lane_marks = placemarks(LANES_FILE, "Mövcud zolaqlar")
parking_marks = placemarks(PARKING_FILE, "velosiped məntəqələri yeni")
lanes, empty_lanes = [], []
for mark in lane_marks:
    name = (mark.findtext("k:name", default="", namespaces=NS) or "").strip()
    parts = [coordinates(node) for node in mark.findall(".//k:LineString/k:coordinates", NS)]
    valid = [part for part in parts if len(part) >= 2]
    if not valid:
        empty_lanes.append(name)
    for part in valid:
        lanes.append((name, [utm39(lon, lat) for lon, lat in part]))

# Mənbədə "8 Noyabr pr." adlı bir xəttin geometriyası boşdur; ölçüyə daxil edilmir.
assert len(lane_marks) == 61 and len(lanes) == 60
assert empty_lanes == ["8 Noyabr pr."]

parking = []
for mark in parking_marks:
    name = (mark.findtext("k:name", default="", namespaces=NS) or "").strip()
    points = [coordinates(node) for node in mark.findall(".//k:Point/k:coordinates", NS)]
    if len(points) != 1 or len(points[0]) != 1:
        raise ValueError(f"{name}: bir nöqtə gözlənilir.")
    lon, lat = points[0][0]
    parking.append((name, lon, lat, utm39(lon, lat)))
assert len(parking) == 185 and len({row[0] for row in parking}) == 185


### Sual 1.1


In [2]:
street_lengths = defaultdict(float)
for street, line in lanes:
    street_lengths[street] += sum(hypot(b[0] - a[0], b[1] - a[1])
                                  for a, b in zip(line, line[1:]))
total_km = sum(street_lengths.values()) / 1000
longest_street = max(street_lengths, key=street_lengths.get)
print(f"{total_km:.2f}, {longest_street}")


57.14, 8 Noyabr pr.


### Sual 1.2


In [3]:
def segment_distance(point, a, b):
    dx, dy = b[0] - a[0], b[1] - a[1]
    length_sq = dx * dx + dy * dy
    if length_sq == 0:
        return hypot(point[0] - a[0], point[1] - a[1])
    fraction = max(0.0, min(1.0, ((point[0] - a[0]) * dx
                                  + (point[1] - a[1]) * dy) / length_sq))
    return hypot(point[0] - a[0] - fraction * dx,
                 point[1] - a[1] - fraction * dy)

def line_distance(point, line):
    return min(segment_distance(point, a, b) for a, b in zip(line, line[1:]))

distance_to_lane = {name: min(line_distance(point, line) for _, line in lanes)
                    for name, _, _, point in parking}
within_50m = sum(distance <= 50.0 for distance in distance_to_lane.values())
print(within_50m)


87


### Sual 1.3


In [4]:
outside = [name for name, lon, lat, _ in parking
           if not (40.30 <= lat <= 40.46 and 49.70 <= lon <= 50.05)]
print(f"{len(outside)}: {', '.join(outside)}")


4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


### Sual 4


In [5]:
recommendation = (
    "Yeni velosiped zolağını Xocalı prospektində mövcud zolağın şərqə davamı kimi "
    "təklif edirəm. MPV-77-2, MPV-64-2 və MPV-65-2 həmin istiqamətdə yerləşir və "
    "indiki zolaqdan 50 metrdən uzaqdır. Mövcud zolağın sonundan başlamaq yeni hissəni "
    "şəbəkəyə birbaşa bağlamağa imkan verər. Bu prospektdə artıq zolaq olduğuna görə "
    "davamının layihələndirilməsi məntiqlidir. Parkinqlərin həqiqətən 50 metr daxilinə "
    "düşməsi, yol eni və kəsişmələr layihə mərhələsində yoxlanmalıdır."
)
assert len(recommendation.split()) <= 99
print(recommendation)


Yeni velosiped zolağını Xocalı prospektində mövcud zolağın şərqə davamı kimi təklif edirəm. MPV-77-2, MPV-64-2 və MPV-65-2 həmin istiqamətdə yerləşir və indiki zolaqdan 50 metrdən uzaqdır. Mövcud zolağın sonundan başlamaq yeni hissəni şəbəkəyə birbaşa bağlamağa imkan verər. Bu prospektdə artıq zolaq olduğuna görə davamının layihələndirilməsi məntiqlidir. Parkinqlərin həqiqətən 50 metr daxilinə düşməsi, yol eni və kəsişmələr layihə mərhələsində yoxlanmalıdır.
